In [ ]:
import json
import os
import time
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import jax
import jax.numpy as jnp
from flax.training import checkpoints
import numpy as np

from experiments.mappings import NEW_MAPPING
from examples.utils import read_utils

from serl_launcher.agents.continuous.bc import BCAgent
from gymnasium.wrappers.record_episode_statistics import RecordEpisodeStatistics

from agentlace.data.data_store import QueuedDataStore
from serl_launcher.utils.launcher import make_bc_agent

from transforms3d.quaternions import quat2mat
import importlib
import cv2
import plotly.graph_objects as go
import plotly.io as pio

from collections import deque
import copy


importlib.reload(read_utils)

pio.renderers.default = 'notebook_connected'
devices = jax.local_devices()
num_devices = len(devices)
sharding = jax.sharding.PositionalSharding(devices)

# base_path = '/home/qiangqiang/workspaces/data/2025-4-3/test_data/ball_pick_teleop_2025_04_02_01'
# json_path = os.path.expanduser('/home/qiangqiang/workspaces/data/2025-4-3/test_data/ ball_pick_teleop_2025_04_02_01/clip_marks.json')
robot_urdf_path = "/home/qiangqiang/workspaces/HK_TACTEXO_DATA/denso_robot_with_ati_4.urdf"

checkpoint_path = "experiments/tennis_ball_pick/2025-4-8_bc_run"
exp_name = "tennis_ball_pick"
eval_checkpoint_step = 19000

seed = 42
learner = False
save_video = False

cam_front_translation = [1.2367936975704506, 0.032497565951025945, 0.5359742126690214]
cam_front_quaternion = [0.012480230443529135, 0.27804828390806924, -0.026321127948753298, -0.960125301139054] # [w, x, y, z]
# Convert quaternion to rotation matrix
cam_front_rotation_matrix = quat2mat([cam_front_quaternion[0], cam_front_quaternion[1],
                                      cam_front_quaternion[2], cam_front_quaternion[3]])

camfront2robot = np.eye(4)
camfront2robot[:3, :3] = cam_front_rotation_matrix
camfront2robot[:3, 3] = cam_front_translation
T = np.array([
    [0, 0, 1, 0], # Maps z -> x
    [-1, 0, 0, 0], # Maps -x -> y
    [0, -1, 0, 0], # Maps -y -> z
    [0, 0, 0, 1],  # Homogeneous coordinate unchanged
 ])

camfront2robot = camfront2robot @ T

robot2camfront = np.linalg.inv(camfront2robot)
fx = 385.732666015625
fy = 385.2535705566406
cx = 324.3997497558594
cy = 240.08477783203125



trajectory_points = []

# action_mean = [0.6289433, -0.02980271, 0.2980561]
# action_std = [0.03720388, 0.10612131, 0.07740753]


action_mean = [ 0.6842325, -0.1687563, 0.2942671]
action_std =  [0.10531126, 0.19766986, 0.08467244 ]



def print_green(x):
    return print("\033[92m {}\033[00m".format(x))

def camera_to_pixel(camera_coords, K):
    """
    将相机坐标系中的点转换到像素坐标系
    :param camera_coords: 相机坐标系中的点，形状为 (3,) 或 (N, 3)
    :param K: 相机内参矩阵，形状为 (3, 3)
    :return: 像素坐标系中的点，形状为 (2,) 或 (N, 2)
    """
    if camera_coords.ndim == 1:
        camera_coords = camera_coords.reshape(1, 3)  # 转换为 (1, 3)

    # 投影到图像坐标系
    pixel_coords_homogeneous = K @ camera_coords.T  # (3, N)
    pixel_coords = pixel_coords_homogeneous[:2] / pixel_coords_homogeneous[2]  # 归一化

    if pixel_coords.shape[1] == 1:
        return pixel_coords.flatten()  # 返回 (2,)
    else:
        return pixel_coords.T  # 返回 (N, 2)
    

class ObsHistoryBuffer:
    def __init__(self, obs_horizon=3, image_keys=("front_camera", ), proprio_key="state"):
        self.obs_horizon = obs_horizon
        self.image_keys = image_keys
        self.proprio_key = proprio_key
        self.buffer = deque(maxlen=obs_horizon)

    def reset(self, first_obs):
        self.buffer.clear()
        for _ in range(self.obs_horizon):
            self.buffer.append(copy.deepcopy(first_obs))

    def append(self, obs):
        self.buffer.append(copy.deepcopy(obs))

    def get_stacked_obs(self):
        stacked_obs = {}
        for key in self.image_keys:
            frames = [o[key] for o in self.buffer]  # list of (H, W, 3)
            stacked_obs[key] = np.concatenate(frames, axis=-1)  # (H, W, 9)

        if self.proprio_key is not None:
            vecs = [o[self.proprio_key] for o in self.buffer]  # list of (23,)
            stacked_obs[self.proprio_key] = np.concatenate(vecs, axis=-1)  # (69,)

        return stacked_obs
    
    def get_success_fail_obs(self):
        stacked_obs = {}
        for key in self.image_keys:
            frames = [o[key] for o in self.buffer]  # list of (H, W, 3)
            stacked_obs[key] = np.stack(frames, axis=0)

        if self.proprio_key is not None:
            vecs = [o[self.proprio_key] for o in self.buffer]  # list of (23,)
            stacked_obs[self.proprio_key] = np.stack(vecs, axis=0)  # (69,)

        return stacked_obs


def actor_test(agent, env, sampling_rng):
    time_list = []

    ckpt = checkpoints.restore_checkpoint(
        os.path.abspath(checkpoint_path),
        agent.state,
        step=eval_checkpoint_step,
    )

    print_green(f"Loaded previous checkpoint at step {eval_checkpoint_step}.")

    agent = agent.replace(state=ckpt)

    data = read_utils.read_data(robot_urdf_path, True)

    # obs, _ = env.reset()
    log_file = "classifier_log_in_training.txt"
    global trajectory_points
    trajectory_points = []
    trajectory_points_read = []
    trajectory_colors = []
    trajectory_colors_read = []
    trajectory_robot_coor = []
    
    try:
        with open(log_file, "w") as f:
            count = 0
            count_pixel = 0
            history = ObsHistoryBuffer(obs_horizon=3)
            for data_count in range(len(data)):
                env.unwrapped.set_data_count(data_count)
                obs = data[data_count]["observations"]

                if data_count == 0:
                    history.reset(obs)
                else:
                    history.append(obs)

                stacked_obs = history.get_stacked_obs()

                sampling_rng, key = jax.random.split(sampling_rng)
                actions_read = data[data_count]["observations"]["state"]
                
                actions = agent.sample_actions(
                    observations=jax.device_put(stacked_obs),
                    argmax=False,
                    seed=key
                )
                actions = np.asarray(jax.device_get(actions)).copy()
                # 将机械臂动作转换到相机坐标系
                action_in_robot_frame = np.array([actions[0], actions[1], actions[2], 1])
                action_read_in_robot_frame = np.array([actions_read[0], actions_read[1], actions_read[2], 1])
                # print("action_in_robot_frame = ", action_in_robot_frame)
                # print("action_read_in_robot_frame = ", action_read_in_robot_frame)
                action_in_cam_frame =  robot2camfront @ action_in_robot_frame
                action_read_in_cam_frame =  robot2camfront @ action_read_in_robot_frame
                # 将相机坐标系中的点转换到像素坐标系
                K = np.array([
                    [fx, 0, cx],
                    [0, fy, cy],
                    [0, 0, 1]
                ])
                camera_coords = action_in_cam_frame[:3]  # 取前 3 个元素 (X_c, Y_c, Z_c)
                camera_coords_read = action_read_in_cam_frame[:3]
                # print("camera_coords = ", camera_coords)
                pixel_coords = camera_to_pixel(camera_coords, K)  # 像素坐标系中的点 (u, v)
                pixel_coords_read = camera_to_pixel(camera_coords_read, K)  # 像素坐标系中的点 (u, v)
                # if count_pixel == 0:
                #     print("pixel_coords = ", pixel_coords)
                #     count_pixel = 1


                # 将动作点添加到轨迹中
                # print("pixel_coords = ", pixel_coords)
                trajectory_points.append(pixel_coords)
                trajectory_colors.append('red')
                trajectory_points_read.append(pixel_coords_read)
                trajectory_colors_read.append('blue')
                trajectory_robot_coor.append(action_in_robot_frame)
                f.flush()

    except KeyboardInterrupt:
        print("\nUser interrupted the program. Printing final logs...")
    
    # finally:
    #     # print(f"success rate: {success_counter / eval_n_trajs}")
    #     print(f"average time: {np.mean(time_list)}")
    
     # 显示图像和轨迹的函数
    def display_image_with_trajectory(index):
        original_width, original_height = 640, 480
        target_width, target_height = 640, 480
        scale_x = target_width / original_width
        scale_y = target_height / original_height
        # 加载图像
        image = data[index]["observations"]["front_camera"]
        resized_image = cv2.resize(image, (target_width, target_height))

        # 显示图像
        plt.imshow(resized_image)

        # 绘制所有轨迹点
        # if trajectory_points:
        #     trajectory_points_array = np.array(trajectory_points)
        #     trajectory_colors_array = np.array(trajectory_colors)
        #     # 过滤超出图像尺寸的点
        #     valid_points = []
        #     valid_colors = []
        #     for i, point in enumerate(trajectory_points_array):
        #         u, v = point
        #         new_u, new_v = int(u * scale_x), int(v * scale_y)
        #         # print("point = ", point)
        #         if 0 <= new_u < target_width*2 and 0 <= new_v < target_height*2:  # 图像尺寸为 640x480
        #             valid_points.append((new_u, new_v))
        #             valid_colors.append(trajectory_colors_array[i])
        #     if valid_points:
        #         valid_points = np.array(valid_points)
        #         valid_colors = np.array(valid_colors, dtype='<U10')
        #         if index < len(valid_points):
        #             valid_colors[index] = 'blue'
        #         # valid_colors = [str(color) for color in valid_colors]
        #         print(f"Valid trajectory points: {len(valid_points)}/{len(trajectory_points_array)}")
        #         plt.scatter(valid_points[:, 0], valid_points[:, 1], c=valid_colors, s=10, label='Trajectory')

        #绘制当前帧坐标
        # 网络算出的当前帧坐标点
        if index < len(trajectory_points):
            net_point = trajectory_points[index]
            u_net, v_net = int(net_point[0]*scale_x), int(net_point[1]*scale_y)
            plt.scatter(u_net, v_net, c='blue', s=50, label='Network Prediction')

        # Ground Truth数据的点
        gt_pixel_coords = trajectory_points_read[index][:3]

        u_gt, v_gt = int(gt_pixel_coords[0]*scale_x), int(gt_pixel_coords[1]*scale_y)
        plt.scatter(u_gt, v_gt, c='green', s=50, label='Ground Truth')


        plt.title(f"Frame: {index}")
        plt.axis('off')
        plt.legend()
        plt.show()


    def plot_3d_trajectory(data, trajectory_robot_coor):
        net_points = []
        gt_points = []

        for idx in range(len(data)):
            # Ground Truth
            gt_state = data[idx]["observations"]["state"]
            gt_pos = gt_state[:3]
            gt_points.append(gt_pos)

            # 网络预测点（逆过程估计深度）
            # pixel_coord = trajectory_robot_coor[:3]
            # Z_c = gt_pos[2]  # 使用GT的Z坐标确保准确
            # X_c = (pixel_coord[0] - cx) * Z_c / fx
            # Y_c = (pixel_coord[1] - cy) * Z_c / fy
            # net_cam_coord = np.array([X_c, Y_c, Z_c, 1.0])
            # net_robot_coord = camfront2robot @ net_cam_coord
            net_points.append(trajectory_robot_coor[idx][:3])

        net_points = np.array(net_points)
        gt_points = np.array(gt_points)

        fig = go.Figure()

        # 网络预测轨迹
        fig.add_trace(go.Scatter3d(
            x=net_points[:,0], y=net_points[:,1], z=net_points[:,2],
            mode='lines+markers',
            name='Network Prediction',
            marker=dict(size=4, color='blue'),
            line=dict(width=2, color='blue')
        ))

        # Ground Truth轨迹
        fig.add_trace(go.Scatter3d(
            x=gt_points[:,0], y=gt_points[:,1], z=gt_points[:,2],
            mode='lines+markers',
            name='Ground Truth',
            marker=dict(size=4, color='green'),
            line=dict(width=2, color='green')
        ))

        fig.update_layout(scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'),
            title="3D Trajectory Comparison"
        )

        # 直接在Notebook内显示
        fig.show()

    # 在actor_test函数末尾直接调用：
    plot_3d_trajectory(data, trajectory_robot_coor)



    # 在 actor_test函数的最后调用此函数
    interact(display_image_with_trajectory, index=IntSlider(min=0, max=len(data)-1, step=1, value=0))
        

def main():
    global config
    config = NEW_MAPPING[exp_name]()

    assert config.batch_size % num_devices == 0
    # seed
    rng = jax.random.PRNGKey(seed)
    rng, sampling_rng = jax.random.split(rng)

    assert exp_name in NEW_MAPPING, "Experiment folder not found."
    env = config.get_environment(
        fake_env=learner,
        save_video=save_video,
        classifier=True,
    )
    env = RecordEpisodeStatistics(env)

    rng, sampling_rng = jax.random.split(rng)

    sample_obs = env.observation_space.sample()
    print("Sample obs keys and shapes:")
    for k, v in sample_obs.items():
        print(f"{k}: {v.shape}")

    agent: BCAgent = make_bc_agent(
        seed=seed,
        sample_obs=env.observation_space.sample(),
        sample_action=env.action_space.sample(),
        image_keys=config.image_keys,
        encoder_type=config.encoder_type,
        action_mean=action_mean,
        action_std=action_std
    )

    # replicate agent across devices
    # need the jnp.array to avoid a bug where device_put doesn't recognize primitives
    agent = jax.device_put(
        jax.tree_map(jnp.array, agent), sharding.replicate()
    )

    if checkpoint_path is not None and os.path.exists(checkpoint_path):
        # input("Checkpoint path already exists. Press Enter to resume training.")
        ckpt = checkpoints.restore_checkpoint(
            os.path.abspath(checkpoint_path),
            agent.state,
        )
        agent = agent.replace(state=ckpt)
        # ckpt_number = os.path.basename(
        #     checkpoints.latest_checkpoint(os.path.abspath(checkpoint_path))
        # )[11:]
        # print_green(f"Loaded previous checkpoint at step {ckpt_number}.")
    
    sampling_rng = jax.device_put(sampling_rng, sharding.replicate())

    # actor loop
    print_green("starting actor loop")
    actor_test(
        agent,
        env,
        sampling_rng,
    )


if __name__ == "__main__":
    main()


Initialized Denso
The ResNet-10 weights already exist at '/home/qiangqiang/.serl/resnet10_params.pkl'.
Loaded 5.418792M parameters from ResNet-10 pretrained on ImageNet-1K
replaced conv_init in encoder_front_camera
replaced norm_init in encoder_front_camera
replaced ResNetBlock_0 in encoder_front_camera
replaced ResNetBlock_1 in encoder_front_camera
replaced ResNetBlock_2 in encoder_front_camera
replaced ResNetBlock_3 in encoder_front_camera
Sample obs keys and shapes:
front_camera: (3, 240, 320, 3)
state: (3, 23)
The ResNet-10 weights already exist at '/home/qiangqiang/.serl/resnet10_params.pkl'.
Loaded 5.418792M parameters from ResNet-10 pretrained on ImageNet-1K
replaced conv_init in pretrained_encoder
replaced norm_init in pretrained_encoder
replaced ResNetBlock_0 in pretrained_encoder
replaced ResNetBlock_1 in pretrained_encoder
replaced ResNetBlock_2 in pretrained_encoder
replaced ResNetBlock_3 in pretrained_encoder


/tmp/ipykernel_886150/2836763049.py:392: DeprecationWarning:

jax.tree_map is deprecated: use jax.tree.map (jax v0.4.25 or newer) or jax.tree_util.tree_map (any JAX version).



 starting actor loop


 Loaded previous checkpoint at step 19000.


ScopeParamShapeError: Initializer expected to generate shape (576, 512) but got shape (320, 512) instead for parameter "kernel" in "/modules_actor/network/Dense_0". (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.ScopeParamShapeError)